# ДЗ-2. Витрина клиента

Вы — аналитик ПРАЙМ, и у вас свой **сегмент клиентов**: его прислал бот 26 сентября, он же лежит в вашей строке `prime.assignments`, в колонке `client_filter`. Окно фактов у всех общее — первое полугодие 2026 года.

Задача недели — собрать **витрину клиента**: таблицу, в которой одна строка — один клиент, а рядом всё, что мы про него знаем. Из витрины потом считается всё остальное, поэтому лишние деньги в ней становятся лишними везде.

Идём по лестнице. Сначала вы дописываете один аргумент, как на семинаре, потом целую строку, потом пишете задачу сами, а в конце собираете витрину и доказываете протоколом, что двойного счёта нет.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение, ваш сегмент, выгрузка восьми таблиц | запускаете |
| **2. Уровень 1** | задачи 1–4: не хватает одного аргумента | вписываете аргумент вместо `ЗАПОЛНИТЕ` |
| **3. Уровень 2** | задачи 5–8: не хватает строки | дописываете строку |
| **4. Уровень 3** | задачи 9–12 | пишете код целиком |
| **5. Уровень 4** | задача 13: витрина и протокол сборки | пишете код целиком |
| **6. Итог** | готовый код: ваши ответы и проверка их формата | запускаете перед сдачей |
| **7. Исследование** | две части для тех, кто хочет 9 или 10 | по желанию |

После каждого уровня — вопрос, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N».

**`ЗАПОЛНИТЕ`** — место, куда вписать ваш код. Пока оно не заполнено, ячейка падает с `NameError: name 'ЗАПОЛНИТЕ' is not defined` — так и задумано.

**Оценка.** Части 1–6 дают до 8 баллов: сделали всё аккуратно — 8, и это «отлично». 9 и 10 ставятся только за исследование в части 7. Все критерии и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-2-витрина-клиента).

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей выполните **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в вашем репозитории как `notebooks/hw2.ipynb`, JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно: запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Подключение | подключается к базе через ваш `.env` и заводит функцию `q(sql)` |
| 1.2 Ваш сегмент | достаёт условие `client_filter` из вашей строки `prime.assignments` |
| 1.3 Выгрузка | загружает восемь таблиц — дальше всё в pandas |
| 1.4 Формат ответов | заводит служебную функцию для итога |

Весь SQL задания собран здесь, в готовом коде. Дальше вы работаете только в pandas.

### 1.1 Подключение

Функция `q(sql)` отправляет запрос в базу и возвращает `DataFrame`, как на семинарах. Соединение открывается на каждый запрос и сразу закрывается.

Если здесь ошибка `Пакет prime не найден`, ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN`, в корне `prime-monitor` нет файла `.env`.

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

### 1.2 Ваш сегмент

Сегмент — это клиенты, которые встречаются в вашей таблице с вашим признаком за ваш период. Бот прислал его вам 26 сентября готовым условием для `where`, оно же лежит в вашей строке `prime.assignments`.

**Эту ячейку не меняйте.** При проверке преподаватель подставит сюда условие **другого** сегмента, и все ответы должны пересчитаться сами. Поэтому ниже нигде не вписывайте руками ни условие, ни числа своего сегмента.

In [ ]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

print(CLIENT_FILTER)

### 1.3 Выгрузка

Восемь таблиц: первые пять — про ваш сегмент, последние три — справочник тарифов и данные всей базы.

| Переменная | Одна строка — это | Что внутри |
|---|---|---|
| `clients` | клиент сегмента | `client_id`, дата регистрации, регион, канал привлечения |
| `subs` | подписка, которой владеет клиент сегмента | тариф, даты начала и конца, пробная ли, причина отмены |
| `pay` | попытка списания по этим подпискам за окно фактов | сумма, **статус**, способ оплаты |
| `sub_members` | участник подписки из `subs` | `subscription_id` и `member_id` — кто подключён к подписке |
| `members` | участие клиента сегмента в любой подписке | `subscription_id` и `client_id` — клиент сегмента как участник |
| `tariffs` | цена тарифа в интервале дат | весь справочник, пять строк |
| `spend` | расходы канала за день | `channel`, `spend_date`, `amount` |
| `registrations` | канал × день регистрации | `n_clients` — сколько клиентов всей базы пришло из канала в этот день |

**Окно фактов общее для всех:** с 1 января по 30 июня 2026 года включительно. В запросе правый конец не включён: `paid_at < '2026-07-01'`.

Самые большие сегменты выгружаются около минуты и занимают в памяти до 2 ГБ — закройте лишние вкладки браузера. Если выгрузка идёт дольше двух минут, сервер оборвёт запрос — что делать, написано в тексте задания, раздел «Если что-то не работает».

In [ ]:
WINDOW = ("2026-01-01", "2026-07-01")   # окно фактов: первое полугодие 2026, правый конец не включён
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= :a and p.paid_at < :b
    order by p.payment_id
""", a=WINDOW[0], b=WINDOW[1])

sub_members = q(f"""
    select m.subscription_id, m.client_id as member_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")

N = len(clients)
print("ЗЕРНО витрины: одна строка — один клиент сегмента. Клиентов:", N)
print()
for name, frame in [("clients", clients), ("subs", subs), ("pay", pay), ("sub_members", sub_members),
                    ("members", members), ("tariffs", tariffs), ("spend", spend),
                    ("registrations", registrations)]:
    print(f"{name:<14} {len(frame):>9} строк")

### 1.4 Формат ответов

Служебная функция: по ней итог в части 6 проверяет, что ответ записан в нужном формате — целое, число с копейками, список. **Верно ли посчитано, она не знает и не сообщает**: это проверяет преподаватель. Код ячейки свёрнут, просто запустите её.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 2 знаков — рубли",
    2: "int",
    3: "int",
    4: "float до 2 знаков — рубли (0.0, если таких клиентов нет)",
    5: "float до 2 знаков — рубли",
    6: "float до 2 знаков — рубли",
    7: "int",
    8: "float до 1 знака — проценты",
    9: "int",
    10: "list [float до 2 знаков, int]",
    11: "list [float до 2 знаков, int]",
    12: "list из 3 int",
    13: "list из 3 int — то, что вернула print_protocol()",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:   # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def pct(x) -> bool:     # float, у которого не больше одного знака после запятой
        return type(x) is float and round(x, 1) == x

    def whole(x) -> bool:
        return type(x) is int

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: money, 2: whole, 3: whole, 4: money, 5: money, 6: money, 7: whole, 8: pct,
        9: whole,
        10: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        11: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        12: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
        13: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Уровень 1 — дыра в один аргумент

Как на семинаре: код готов, не хватает одного аргумента. Впишите его вместо `ЗАПОЛНИТЕ` и запустите ячейку.

Последняя строка каждой задачи кладёт ответ в `answer_N`. Эти строки и имена `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом сегменте.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All. Замените всё содержимое ячейки одной строкой `answer_N = None`, где N — номер задачи: итог покажет «❌ не решена», а остальное проверится как обычно.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 6 проверяет только формат. Правильность проверяет преподаватель — на вашем сегменте и на контрольном.

**В `pay` лежат все попытки списания:** `success`, `failed` и `refunded`. Какие из них деньги, решаете вы — это задача 1.

### Задача 1. Контрольное число ДЕНЬГИ

Первое правило сборки витрины: контрольное число считают до сборки. Мы заранее знаем, сколько денег принёс сегмент за окно, — и потом витрина обязана дать ровно столько же.

**Что сделать**

Оставьте в `pay` только платежи, которые действительно принесли деньги, и посчитайте их сумму.

**Формат ответа:** `float` до двух знаков, рубли — например `98765432.1`.

**Подсказка:** Какие статусы бывают, покажет `pay["status"].value_counts()`.

In [ ]:
# Какие попытки списания — это деньги, которые остались у компании?
success = pay[pay["status"] == "success"]
revenue = success["amount"].sum()

print("платежей с этим статусом:", len(success))
answer_1 = round(float(revenue), 2)
answer_1

### Задача 2. Подписки без платежа

Руководитель спрашивает, сколько подписок сегмента за полгода не принесли ни рубля. Чтобы ответить, к каждой подписке присоединяют число её успешных платежей и смотрят, у кого пусто.

**Что сделать**

Выберите вид соединения так, чтобы подписки без платежей **остались** в результате — с пропуском в `n_payments`.

**Формат ответа:** `int` — например `4321`.

**Подсказка:** Слайд 7 Л4: `how=` решает судьбу строк, которым не нашлась пара.

In [ ]:
# Сколько успешных платежей пришло по каждой подписке за окно.
by_sub = (pay[pay["status"] == "success"]
          .groupby("subscription_id", as_index=False)
          .agg(n_payments=("payment_id", "size")))

# Подписка без платежей должна остаться в результате — с пропуском в n_payments.
subs_pay = subs.merge(by_sub, on="subscription_id", how="left", validate="1:1")

no_payment = subs_pay["n_payments"].isna().sum()
print("подписок:", len(subs), " строк после соединения:", len(subs_pay))
answer_2 = int(no_payment)
answer_2

### Задача 3. Кто платил

Сколько владельцев из сегмента заплатили за полгода хотя бы раз? Для этого к каждому платежу приписывают владельца подписки. Но прежде чем соединять, объявляют кратность: сколько платежей приходится на одну подписку и сколько подписок — на один платёж.

**Что сделать**

Впишите в `validate=` кратность связи «платёж → подписка».

**Формат ответа:** `int` — например `5432`.

**Подсказка:** Слайд 6 Л4, четыре случая кратности. Если слева на один ключ приходится много строк, это «m».

In [ ]:
success = pay[pay["status"] == "success"]

# Сколько платежей приходится на одну подписку — и сколько подписок на один платёж?
pay_owner = success.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id", how="left", validate="m:1",
)

payers = pay_owner["owner_client_id"].nunique()
print("платежей до соединения:", len(success), " после:", len(pay_owner))
answer_3 = int(payers)
answer_3

### Задача 4. Выручка без канала

Маркетинг смотрит выручку по каналам привлечения. У части клиентов канал не записан, и если этого не заметить, сумма по каналам тихо разойдётся с общей выручкой.

**Что сделать**

Сделайте так, чтобы клиенты без канала попали в отчёт отдельной строкой, и посчитайте их выручку.

**Формат ответа:** `float` до двух знаков, рубли — например `123456.7`. Если таких клиентов в сегменте нет — `0.0`.

**Подсказка:** Слайд 18 Л4: сумма по группам не сходится с общей. Сверьте сумму по каналам с выручкой задачи 1.

In [ ]:
success = pay[pay["status"] == "success"]
pay_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="m:1"))

# Клиенты без канала должны остаться в отчёте отдельной строкой.
by_channel = pay_client.groupby("acquisition_channel", dropna=False)["amount"].sum()

print(by_channel.round(2).to_string())
no_channel = by_channel[by_channel.index.isna()].sum()
answer_4 = round(float(no_channel), 2)
answer_4

**Вопрос 1.** В `pay` есть попытки списания со статусом `refunded`, и сумма у них положительная. Сколько таких строк в вашем сегменте и на какую сумму? Почему эту сумму нельзя прибавить к выручке — и почему её нельзя вычесть из выручки?

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.

refunded = pay[pay["status"] == "refunded"]
refunded_sum = round(float(refunded["amount"].sum()), 2)

print("строк со статусом refunded:", len(refunded))
print("сумма по ним, руб.:", refunded_sum)
print("для масштаба — выручка задачи 1, руб.:", answer_1,
      f"(возвраты — {refunded_sum / answer_1:.2%} от неё)")
# Попала ли хоть одна такая строка в выручку задачи 1? Там фильтр status == "success".
print("refunded среди строк выручки задачи 1:", int((success["status"] == "refunded").sum()))

#### ✍️ Ответ на вопрос 1

В сегменте 3 003 строки со статусом refunded, общая сумма возвратов — 925 297 руб. (это где-то 1,27% от нашей выручки в 72,6 млн руб. из первой задачи).
Прибавлять эти деньги к выручке не имеет смысла, поскольку их уже вернули клиентам, физически у компании их нет. Вычитать их из рассчитанной выручки (72,6 млн руб.) тоже нельзя, поскольку туда попали строго успешные транзакции (success), и строк refunded там просто нет. Если мы их вычтем, то спишем потерю дважды.
В отчёте лучше всего просто вывести сумму возвратов отдельной строкой или метрикой, чтобы финансы видели объём потерь, но саму выручку при этом корректировать не нужно.


## Часть 3. Уровень 2 — дыра в строку

Теперь не хватает целой строки: выражения, условия или набора аргументов. Всё остальное готово. Правила те же: строки с `answer_N` не трогайте, не получилось — `answer_N = None`.

### Задача 5. Свернуть до клиента

Средний доход с платящего клиента — ARPPU — это выручка, делённая на число клиентов, которые платили. Чтобы посчитать его по витрине, платежи сначала сворачивают до зерна «клиент» и только потом соединяют с клиентами — иначе соединение перемножит строки.

**Что сделать**

Допишите `agg(...)`: для каждого владельца — `n_payments` (сколько успешных платежей) и `paid` (сколько он заплатил).

**Формат ответа:** `float` до двух знаков, рубли — например `1234.56`.

**Подсказка:** Слайд 17 Л4, именованные агрегаты: `agg(имя=("колонка", "функция"), ...)`.

In [ ]:
success = pay[pay["status"] == "success"]

# Одна строка — один владелец: сколько успешных платежей и сколько денег.
by_client = (success
             .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
             .groupby("owner_client_id", as_index=False)
             .agg(n_payments=("payment_id", "size"), paid=("amount", "sum")))

mart5 = clients.merge(by_client, left_on="client_id", right_on="owner_client_id",
                      how="left", validate="1:1")

paying = mart5["n_payments"].notna().sum()
arppu = mart5["paid"].sum() / paying
print("клиентов:", len(mart5), " из них платили:", paying)
answer_5 = round(float(arppu), 2)
answer_5

### Задача 6. Цена по справочнику

В `tariffs` лежит история цен: у тарифа несколько строк с интервалами действия, а одна строка ошибочная — вы нашли её на семинаре. Сколько принёс тариф «Расширенный» по цене справочника?

**Что сделать**

Допишите две строки: 1) справочник без ошибочной строки; 2) условие «момент платежа попадает в интервал действия цены» по конвенции курса.

**Формат ответа:** `float` до двух знаков, рубли — например `2345678.9`.

**Подсказка:** Раздел 2 семинара С4. Конвенция границы — `(paid_at >= valid_from) & (paid_at < valid_to + 1 день)`, потому что `valid_to` — это дата, то есть полночь. Каждый платёж должен найти в справочнике ровно одну цену: сравните число строк до и после.

In [ ]:
# Даты справочника — дни без часового пояса, а платежи — моменты в UTC.
ref = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")

# 1. Справочник без строки, из-за которой интервалы одного тарифа перекрываются.
# Строка 429 руб. «Расширенного» перекрывает и старую, и новую цену (семинар С4, раздел 2).
ref_fixed = ref[ref["monthly_fee"] != 429]

pay_t = (pay[pay["status"] == "success"]
         .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
m = pay_t.merge(ref_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]],
                on="tariff_id")

# 2. Момент платежа попадает в интервал действия цены — по конвенции курса.
hit = (m["paid_at"] >= m["valid_from"]) & (m["paid_at"] < m["valid_to"] + pd.Timedelta(days=1))
priced = m[hit]

print("платежей:", len(pay_t), " строк с ценой:", len(priced))
extended = priced.loc[priced["tariff_name"] == "Расширенный", "monthly_fee"].sum()
answer_6 = round(float(extended), 2)
answer_6

### Задача 7. Выше среднего по своему каналу

Маркетинг хочет знать, сколько платящих клиентов приносят больше, чем типичный клиент **их же** канала. Для этого рядом с каждым клиентом ставят среднее по его каналу — не сворачивая таблицу.

**Что сделать**

Допишите строку: `channel_avg` — среднее `paid` по каналу клиента, по значению в каждой строке.

**Формат ответа:** `int` — например `6543`.

**Подсказка:** Слайд 19 Л4: `transform` возвращает столько же строк, сколько было.

In [ ]:
success = pay[pay["status"] == "success"]
per_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .groupby("owner_client_id", as_index=False)
              .agg(paid=("amount", "sum"))
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="1:1"))
per_client = per_client[per_client["acquisition_channel"].notna()]   # только известные каналы

# Рядом с каждым клиентом — среднее paid по его каналу. Строк остаётся столько же.
per_client["channel_avg"] = per_client.groupby("acquisition_channel")["paid"].transform("mean")

above = (per_client["paid"] > per_client["channel_avg"]).sum()
print("платящих клиентов с известным каналом:", len(per_client))
answer_7 = int(above)
answer_7

### Задача 8. Выручка по месяцам

Финансам нужна таблица: строки — месяцы, колонки — тарифы, в клетках — выручка. По ней видно, как выручка «Базового» менялась от месяца к месяцу.

**Что сделать**

Допишите аргументы `pivot_table`: строки — `month`, колонки — `tariff_id`, значения — сумма `amount`.

**Формат ответа:** `float` до одного знака, проценты — например `87.6`.

**Подсказка:** Слайд 23 Л4.

In [ ]:
success = (pay[pay["status"] == "success"]
           .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
success["month"] = success["paid_at"].dt.strftime("%Y-%m")

# Строки — месяцы, колонки — тарифы (1 — «Базовый», 2 — «Расширенный»), в клетках — выручка.
by_month = success.pivot_table(index="month", columns="tariff_id", values="amount", aggfunc="sum")

print(by_month.round(2).to_string())
april_to_march = by_month.loc["2026-04", 1] / by_month.loc["2026-03", 1] * 100
answer_8 = round(float(april_to_march), 1)
answer_8

**Вопрос 2.** Выручка «Базового» в апреле — `answer_8` процентов от мартовской. Это отток клиентов? Проверьте по дням: сколько дней апреля есть в `pay` и как выглядит выручка по дням. Объясните, что случилось, и что вы написали бы финансам в отчёте.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.

def rub(x: float) -> str:
    """Рубли с пробелом между разрядами: 1234567 -> «1 234 567»."""
    return f"{x:,.0f}".replace(",", " ")


# Какие дни апреля вообще есть в выгрузке платежей (любой статус)?
pay_day = pay["paid_at"].dt.strftime("%Y-%m-%d")
april_days = set(pay_day[pay_day.str.startswith("2026-04")])
all_april = pd.date_range("2026-04-01", "2026-04-30").strftime("%Y-%m-%d")
missing_april = [day for day in all_april if day not in april_days]
print("дней апреля в pay:", len(april_days), "из 30; нет данных за:", missing_april)

# Выручка «Базового» по дням: сравниваем не месяцы целиком, а средний день.
basic = (pay[pay["status"] == "success"]
         .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
basic = basic[basic["tariff_id"] == 1]
basic_by_day = basic.groupby(basic["paid_at"].dt.strftime("%Y-%m-%d"))["amount"].sum()
march = basic_by_day[basic_by_day.index.str.startswith("2026-03")]
april = basic_by_day[basic_by_day.index.str.startswith("2026-04")]
print(f"март:   дней в данных {len(march)}, в среднем {rub(march.mean())} руб. в день")
print(f"апрель: дней в данных {len(april)}, в среднем {rub(april.mean())} руб. в день")
print(f"средний день апреля к среднему дню марта: {april.mean() / march.mean() * 100:.1f} %")
print()
print("выручка «Базового» по дням вокруг дыры:")
print(april.loc["2026-04-10":"2026-04-19"].round(2).to_string())

#### ✍️ Ответ на вопрос 2

Результат в 86,2% пугает, но это не реальный отток. Дело в данных: если посмотреть в pay, то за апрель там не хватает двух дней, 14 и 15 апреля отсутствуют полностью. Плюс, в марте 31 день, а в апреле 30. Если считать правильно, по среднему дню, то в марте мы получали 209 007 руб./день, а в апреле — 199 405 руб./день. Это 95,4% от мартовских показателей, а оставшаяся просадка в примерно 4,6% похожа на наш естественный отток (около 3,9% в месяц, как видно в исследовании Б). У нас закрытый сегмент, новые юзеры не приходят, так что они постепенно уменьшаются по количеству.
Финансам я бы написал: всегда нормировать выручку по дням, за которые реально есть данные, и явно подсвечивать пропуски (типа «нет логов за 14–15 апреля»), иначе можно сделать ложные выводы об обвале выручки.


## Часть 4. Уровень 3 — пишете сами

Заготовок больше нет: в ячейке только условие в комментарии и строка с `answer_N`. Код вы пишете целиком — от загруженных таблиц до ответа. Печатайте промежуточные результаты: так проще найти ошибку и объяснить решение на защите.

Каждая задача опирается на приём с лекции Л4 и семинара С4 — номер слайда стоит в подсказке. Ответ кладите в `answer_N` обычным `int` или `float`, а не `numpy`-числом: `int(...)`, `round(float(...), 2)`.

### Задача 9. Люди, а не места

Сколько людей сегмента пользуются ПРАЙМ? Пользуется тот, кто владеет подпиской или участвует хотя бы в одной чужой. Один человек может участвовать в нескольких подписках: если не свернуть участия до клиента, вы насчитаете места, а не людей.

**Что сделать**

Посчитайте клиентов сегмента, которые владеют подпиской (`subs`) или участвуют хотя бы в одной (`members`). Одна строка — один клиент.

**Формат ответа:** `int` — например `7654`.

**Подсказка:** Раздел 3 семинара С4: свернуть участия до клиента, соединить с `validate="1:1"`, флаг «владелец или участник».

In [ ]:
# Задача 9. Сколько клиентов сегмента пользуются ПРАЙМ: владеют подпиской
# или участвуют хотя бы в одной. Одна строка — один клиент. Ответ — обычный int.

# Сворачиваем обе стороны «многие» до зерна «клиент» ДО соединения.
owners = (subs.groupby("owner_client_id", as_index=False)
          .agg(n_owned=("subscription_id", "size")))            # сколько подписок у владельца
memberships = (members.groupby("client_id", as_index=False)
               .agg(n_memberships=("subscription_id", "size")))  # в скольких подписках участник

users = (clients[["client_id"]]
         .merge(owners, left_on="client_id", right_on="owner_client_id", how="left", validate="1:1")
         .merge(memberships, on="client_id", how="left", validate="1:1"))

# Пользуется ПРАЙМ тот, у кого есть своя подписка ИЛИ хотя бы одно участие.
users["uses_prime"] = users["n_owned"].notna() | users["n_memberships"].notna()

print("клиентов в сегменте:", len(users), " строк после соединений:", len(users))
print("владельцев:", int(users["n_owned"].notna().sum()),
      " участников:", int(users["n_memberships"].notna().sum()),
      " мест участия до свёртки:", len(members))

answer_9 = int(users["uses_prime"].sum())
answer_9

### Задача 10. Сколько человек на подписку

Подпиской пользуются владелец и те, кого он подключил. Подключить можно не больше `share_slots` человек — это поле справочника тарифов. Сколько в среднем человек приходится на подписку сегмента и сколько подписок заполнены до предела?

**Что сделать**

1. Для каждой подписки из `subs`: один владелец плюс число её строк в `sub_members`.
2. Среднее по всем подпискам сегмента — первая часть ответа.
3. Присоедините к подпискам `share_slots` из `tariffs`. Подписок, где людей ровно `share_slots` + 1, — вторая часть ответа.
4. Напечатайте, у скольких подписок людей больше, чем `share_slots` + 1. Это инвариант: число обязано быть нулём.

**Формат ответа:** `[среднее, заполненных до предела]` — например `[3.21, 987]`: `float` до двух знаков и `int`.

**Подсказка:** Слайды 8 и 14 Л4. У одного `tariff_id` в справочнике несколько строк, и `validate="m:1"` об этом скажет. Какая часть справочника на самом деле нужна для `share_slots`?

In [ ]:
# Задача 10. Люди на подписках сегмента, считая владельца, и лимит share_slots.
# Ответ — [среднее на подписку (float до 2 знаков), заполненных до предела (int)].

# 1. Число участников каждой подписки — сворачиваем sub_members до зерна «подписка».
per_sub = (sub_members.groupby("subscription_id", as_index=False)
           .agg(n_members=("member_id", "size")))
people = subs[["subscription_id", "tariff_id"]].merge(per_sub, on="subscription_id",
                                                      how="left", validate="1:1")
# Подписка без участников осталась с пропуском — у неё только владелец.
people["people"] = 1 + people["n_members"].fillna(0).astype("int64")

# 2. share_slots не меняется с историей цен: из справочника нужна только пара
#    tariff_id -> share_slots, по одной строке на тариф. Иначе validate="m:1" упадёт.
slots = tariffs[["tariff_id", "share_slots"]].drop_duplicates()
people = people.merge(slots, on="tariff_id", how="left", validate="m:1")

full = int((people["people"] == people["share_slots"] + 1).sum())
over = int((people["people"] > people["share_slots"] + 1).sum())
print("подписок:", len(people), " людей на подписку:", people["people"].value_counts().sort_index().to_dict())
print("заполнены до предела:", full)
print("людей больше лимита (инвариант, обязан быть 0):", over)

answer_10 = [round(float(people["people"].mean()), 2), full]
answer_10

### Задача 11. Сколько стоил клиент

CAC — сколько стоило привести одного клиента. Расходы на привлечение известны по каналу и дню, а не по клиенту: присоединить их к витрине напрямую нельзя. Поэтому CAC считают на его собственном зерне — «канал × месяц» — и только потом приписывают клиентам.

**Что сделать**

1. CAC канала за месяц — это расходы канала за месяц (`spend`), делённые на число всех клиентов базы, которые пришли из этого канала в этом месяце (`registrations`). Берите только полные месяцы расходов — с 2025-02 по 2026-07.
2. Каждому клиенту сегмента припишите CAC его канала в месяц его регистрации.
3. Посчитайте средний CAC по клиентам сегмента, у которых он определён, и число клиентов, у которых он не определён: канал не записан или регистрация вне этих месяцев.

**Формат ответа:** `[средний CAC, клиентов без CAC]` — например `[432.1, 876]`: `float` до двух знаков и `int`.

**Подсказка:** Слайд 28 Л4: `marketing_spend` не присоединяется к витрине — у него другое зерно. Месяц из даты: `pd.to_datetime(...).dt.strftime("%Y-%m")`. Соединение по двум колонкам: `on=["channel", "month"]`, и кратность объявите.

In [ ]:
# Задача 11. CAC по каналу и месяцу, затем средний CAC клиентов сегмента.
# Ответ — [средний CAC (float до 2 знаков), клиентов без CAC (int)].

# 1. Расходы — зерно «канал × день» -> сворачиваем до «канал × месяц».
spend_m = spend.assign(month=pd.to_datetime(spend["spend_date"]).dt.strftime("%Y-%m"))
spend_m = spend_m.groupby(["channel", "month"], as_index=False).agg(spend=("amount", "sum"))

# Регистрации всей базы — тоже до «канал × месяц». Строки без канала group by выбросит,
# и это правильно: расходов на «неизвестный канал» не бывает.
reg_m = registrations.assign(month=pd.to_datetime(registrations["reg_date"]).dt.strftime("%Y-%m"))
reg_m = reg_m.groupby(["channel", "month"], as_index=False).agg(n_clients=("n_clients", "sum"))

# CAC живёт на своём зерне «канал × месяц»; берём только полные месяцы расходов.
cac = spend_m.merge(reg_m, on=["channel", "month"], how="inner", validate="1:1")
cac = cac[(cac["month"] >= "2025-02") & (cac["month"] <= "2026-07")].copy()
cac["cac"] = cac["spend"] / cac["n_clients"]
print("строк CAC (канал × месяц):", len(cac), " каналов:", cac["channel"].nunique())

# 2. Каждому клиенту — CAC его канала в месяц его регистрации.
client_cac = clients[["client_id", "acquisition_channel", "registered_at"]].assign(
    month=clients["registered_at"].dt.strftime("%Y-%m"))
client_cac = client_cac.merge(cac[["channel", "month", "cac"]],
                              left_on=["acquisition_channel", "month"], right_on=["channel", "month"],
                              how="left", validate="m:1")
print("строк до:", len(clients), " после:", len(client_cac))

# 3. Средний CAC там, где он определён, и сколько клиентов без CAC.
no_cac = int(client_cac["cac"].isna().sum())
print("без CAC:", no_cac, " из них без канала:", int(client_cac["acquisition_channel"].isna().sum()))

answer_11 = [round(float(client_cac["cac"].mean()), 2), no_cac]
answer_11

### Задача 12. Лучшие регионы канала

Где продавать ПРАЙМ дальше? Маркетинг просит три лучших региона по выручке в каждом канале привлечения — и начинает с самого доходного канала.

**Что сделать**

Посчитайте выручку сегмента за окно по каналу и региону владельца; клиенты без канала не участвуют. В каждом канале отберите три региона с наибольшей выручкой. Ответ — три `region_code` самого доходного канала сегмента, от большей выручки к меньшей.

**Формат ответа:** список из трёх `int` — например `[12, 34, 56]`.

**Подсказка:** Слайд 25 Л4: «три лучших вообще» и «три лучших в каждом канале» — разные вещи.

In [ ]:
# Задача 12. Три лучших региона в каждом канале; ответ — для самого доходного канала.
# Ответ — список из трёх int, от большей выручки к меньшей.

paid12 = (pay[pay["status"] == "success"]
          .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
          .merge(clients[["client_id", "acquisition_channel", "region_code"]],
                 left_on="owner_client_id", right_on="client_id", validate="m:1"))
paid12 = paid12[paid12["acquisition_channel"].notna()]     # клиенты без канала не участвуют

by_region = (paid12.groupby(["acquisition_channel", "region_code"], as_index=False)
             .agg(revenue=("amount", "sum")))

# Топ-3 В КАЖДОМ канале: сортируем внутри канала по убыванию выручки и берём голову группы.
# При равной выручке раньше идёт меньший region_code — чтобы ответ был однозначным.
top3 = (by_region.sort_values(["acquisition_channel", "revenue", "region_code"],
                              ascending=[True, False, True])
        .groupby("acquisition_channel").head(3))

best_channel = by_region.groupby("acquisition_channel")["revenue"].sum().idxmax()
print("самый доходный канал:", best_channel)
print(top3[top3["acquisition_channel"] == best_channel].round(2).to_string(index=False))

answer_12 = [int(r) for r in top3.loc[top3["acquisition_channel"] == best_channel, "region_code"]]
answer_12

**Вопрос 3.** Куда вы добавили бы рубль на привлечение? Сравните каналы по двум числам: CAC канала (задача 11) и выручка на клиента вашего сегмента из этого канала. Назовите канал и оба числа. Почему выручку вашего сегмента нельзя просто разделить на CAC канала — какую оговорку нужно сделать?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.

# CAC канала за весь период из задачи 11: все расходы канала / все пришедшие клиенты.
channel_cac = cac.groupby("channel").agg(spend=("spend", "sum"), n_clients=("n_clients", "sum"))
channel_cac["cac"] = channel_cac["spend"] / channel_cac["n_clients"]

# Выручка сегмента за окно на одного клиента сегмента из этого канала.
seg_revenue = (pay[pay["status"] == "success"]
               .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
               .merge(clients[["client_id", "acquisition_channel"]],
                      left_on="owner_client_id", right_on="client_id", validate="m:1")
               .groupby("acquisition_channel")["amount"].sum())
seg_clients = clients.groupby("acquisition_channel").size()

compare = channel_cac[["cac"]].join((seg_revenue / seg_clients).rename("revenue_per_client"))
compare["revenue_to_cac"] = compare["revenue_per_client"] / compare["cac"]
print(compare.sort_values("revenue_to_cac", ascending=False).round(2).to_string())

#### ✍️ Ответ на вопрос 3

Вложился бы в organic, поскольку у него самый низкий CAC (всего 137,45 руб.), при этом средняя выручка на клиента из нашего сегмента составляет 1 585,05 руб., а это окупаемость больше чем в 11 раз. Для сравнения, тот же performance вообще работает в минус, привлекая за 1 532 руб. и получая выручку с клиента в 1 367 руб.
Но надо понимать, что сравнение не совсем адекватное, поскольку CAC считался по всем регистрациям в базе (включая тех, кто ничего не купил), а выручку мы берём только по нашему отборному сегменту платящих юзеров с кошельком. А также выручка здесь за полгода, а не за всё время жизни клиента (LTV). Поэтому метрики перекошены, но органика всё равно выглядит самым безопасным и маржинальным выбором.

## Часть 5. Уровень 4 — витрина и протокол сборки

Главная задача недели. Вы собираете **витрину клиента**: одна строка — один клиент сегмента, рядом всё, что про него известно. И доказываете, что двойного счёта нет, **протоколом сборки** — той же формы, что на слайде 15 Л4 и на семинаре С4.

> **Протокол — это 3 балла из 10.** Без него ноль, даже если все числа верные: результат, который нельзя перепроверить, в работе не принимают.

Три последние строки протокола — инварианты витрины:

- **ЗЕРНО:** строк в витрине столько же, сколько уникальных клиентов, и столько, сколько вы объявили до сборки;
- **ДЕНЬГИ:** сумма по витрине сходится с независимым контролем до копейки;
- **ЛЮДИ:** число тех, кто пользуется ПРАЙМ, сходится со счётом, сделанным другим путём.

Ниже готовые функции, эту ячейку просто запустите. `step()` — та же, что на семинаре: одна строка протокола на одно соединение. `check_step()` делает строку протокола для проверки из вашего `src/prime/checks.py`. `print_protocol()` печатает протокол и возвращает ответ задачи 13.

In [ ]:
def step(name: str, before: pd.DataFrame, after: pd.DataFrame, key: str,
         validate: str = "", expect_lost: int | None = None) -> dict:
    """Одна строка протокола сборки.

    keys_lost — строки слева, которым не нашлось пары справа.
    Видно их по пропуску в колонке key, пришедшей из правой таблицы.
    """
    lost = int(after[key].isna().sum())
    if len(after) != len(before):
        verdict = f"ВНИМАНИЕ: строк было {len(before)}, стало {len(after)}"
    elif expect_lost is None:
        verdict = "ok"
    elif lost != expect_lost:
        verdict = f"ВНИМАНИЕ: потеряно {lost}, ждали {expect_lost}"
    else:
        verdict = f"ok, ожидали {expect_lost}"
    return {"step": name, "rows_before": len(before), "rows_after": len(after),
            "validate": validate, "keys_lost": lost, "verdict": verdict}


def check_step(name: str, frame: pd.DataFrame, bad: pd.DataFrame, expected: int) -> dict:
    """Строка протокола для проверки из checks.py.

    bad — строки, которые нашла проверка; expected — сколько вы ждали, объявив это ДО сборки.
    """
    found = len(bad)
    verdict = f"ok, ожидали {expected}" if found == expected else f"ВНИМАНИЕ: нашлось {found}, ждали {expected}"
    return {"step": name, "rows_before": len(frame), "rows_after": len(frame),
            "validate": "проверка", "keys_lost": found, "verdict": verdict}


def print_protocol(protocol: list[dict], mart: pd.DataFrame, declared: int,
                   money_mart: float, money_control: float,
                   people_mart: int, people_control: int) -> list[int]:
    """Напечатать протокол сборки. Вернуть [строк в витрине, разница ДЕНЬГИ в копейках, ЛЮДИ по витрине]."""
    rows, unique = len(mart), int(mart["client_id"].nunique())
    kopecks_mart, kopecks_control = round(money_mart * 100), round(money_control * 100)
    diff = kopecks_mart - kopecks_control

    print("ПРОТОКОЛ СБОРКИ ВИТРИНЫ: одна строка — один клиент сегмента")
    print()
    print(pd.DataFrame(protocol).to_string(index=False))
    print()
    print(f"ЗЕРНО   строк {rows}, клиентов {unique}, объявляли {declared}"
          f"   -> {'ok' if rows == unique == declared else 'ВНИМАНИЕ'}")
    print(f"ДЕНЬГИ  по витрине {money_mart:.2f} ₽, независимо {money_control:.2f} ₽, разница {diff} коп."
          f"   -> {'ok' if diff == 0 else 'ВНИМАНИЕ'}")
    print(f"ЛЮДИ    по витрине {people_mart}, независимо {people_control}"
          f"   -> {'ok' if people_mart == people_control else 'ВНИМАНИЕ'}")
    print()
    print("HW2_PROTOCOL=" + json.dumps({
        "steps": protocol, "grain": [rows, unique, declared],
        "money_kopecks": [kopecks_mart, kopecks_control], "people": [int(people_mart), int(people_control)],
    }, ensure_ascii=False, default=str))
    return [int(rows), int(diff), int(people_mart)]


print("функции протокола готовы")

### Задача 13. Витрина клиента

Соберите витрину по своему сегменту и докажите протоколом, что двойного счёта нет. Весь раздел 4 семинара С4 — это тот же путь на общем сегменте.

**Что сделать**

1. **До сборки** объявите, сколько строк будет в витрине, и сколько ключей потеряет каждое левое соединение — эти числа пойдут в `expect_lost`.
2. Соберите `mart`, одна строка — один клиент сегмента:
   - `client_id`, `acquisition_channel`, `region_code` — из `clients`;
   - `subscription_id`, `tariff_id` — подписка клиента, пропуск, если её нет;
   - `n_payments`, `paid` — успешные платежи за окно, свёрнутые до клиента; `0`, если их нет;
   - `n_memberships` — в скольких подписках клиент участник; `0`, если ни в одной;
   - `uses_prime` — владелец или участник.
3. Каждое соединение запишите в протокол через `step()`.
4. Примените к витрине функцию из своего `src/prime/checks.py` — например, `find_missing_channel` — и запишите её строкой `check_step()` с ожиданием, объявленным до сборки. Это отдельные 0,5 балла из 10.
5. Посчитайте независимые контроли для ДЕНЬГИ и ЛЮДИ **не по витрине, а другим путём** и вызовите `print_protocol()`.

**Формат ответа:** то, что вернула `print_protocol()`: список из трёх `int`.

**Подсказка:** Если `src/prime/checks.py` у вас нет — создайте его по образцу семинара С3 и закоммитьте в `main`: `from prime.checks import find_missing_channel`. Протокол должен уметь покраснеть: проверьте его, сломав одно соединение, как на семинаре.

In [ ]:
# Задача 13. Витрина клиента и протокол сборки.
# Одна строка — один клиент сегмента. Каждое соединение — через step()
# с ожиданием, объявленным ДО сборки. Проверка из checks.py — через check_step().
# В конце — answer_13 = print_protocol(...).
from prime.checks import find_missing_channel

# ---- 1. ДО сборки: зерно и ожидания по каждому соединению ----
N_DECLARED = len(clients)                                        # одна строка — один клиент сегмента

# Сторона «многие» сворачивается до зерна «клиент» ДО соединения.
success = pay[pay["status"] == "success"]
by_client = (success
             .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
             .groupby("owner_client_id", as_index=False)
             .agg(n_payments=("payment_id", "size"), paid=("amount", "sum")))
memberships = (members.groupby("client_id", as_index=False)          # до КЛИЕНТА, а не до подписки:
               .agg(n_memberships=("subscription_id", "size")))      # иначе посчитаем места, а не людей

EXPECT_NO_SUB = N_DECLARED - subs["owner_client_id"].nunique()       # клиенты без своей подписки
EXPECT_NO_PAY = N_DECLARED - len(by_client)                           # без успешных платежей за окно
EXPECT_NO_MEMBER = N_DECLARED - len(memberships)                      # не участники ни одной подписки
EXPECT_NO_CHANNEL = int(clients["acquisition_channel"].isna().sum())  # ждём от find_missing_channel

print("объявлено ДО сборки:")
print(f"  строк в витрине:            {N_DECLARED}")
print(f"  потеряет клиент->подписка:  {EXPECT_NO_SUB}")
print(f"  потеряет платежи->клиент:   {EXPECT_NO_PAY}")
print(f"  потеряет участия->клиент:   {EXPECT_NO_MEMBER}")
print(f"  найдёт find_missing_channel: {EXPECT_NO_CHANNEL}")
print()

# ---- 2. Сборка: каждое соединение — строка протокола ----
protocol: list[dict] = []

base = clients[["client_id", "acquisition_channel", "region_code"]]
mart = (base.merge(subs[["subscription_id", "owner_client_id", "tariff_id"]],
                   left_on="client_id", right_on="owner_client_id", how="left", validate="1:1")
        .drop(columns="owner_client_id"))
protocol.append(step("клиент -> подписка", base, mart, "subscription_id", "1:1", EXPECT_NO_SUB))

before = mart
mart = (mart.merge(by_client, left_on="client_id", right_on="owner_client_id", how="left", validate="1:1")
        .drop(columns="owner_client_id"))
protocol.append(step("платежи -> клиент", before, mart, "n_payments", "1:1", EXPECT_NO_PAY))

before = mart
mart = mart.merge(memberships, on="client_id", how="left", validate="1:1")
protocol.append(step("участия -> клиент", before, mart, "n_memberships", "1:1", EXPECT_NO_MEMBER))

# Пропуски заполняем ПОСЛЕ строк протокола: после fillna потери уже не видно.
mart["n_payments"] = mart["n_payments"].fillna(0).astype("int64")
mart["paid"] = mart["paid"].fillna(0.0)
mart["n_memberships"] = mart["n_memberships"].fillna(0).astype("int64")
mart["uses_prime"] = mart["subscription_id"].notna() | (mart["n_memberships"] > 0)

# ---- 3. Проверка из src/prime/checks.py — с ожиданием, объявленным выше ----
protocol.append(check_step("нет канала привлечения", mart, find_missing_channel(mart), EXPECT_NO_CHANNEL))

# ---- 4. Независимые контроли — НЕ по витрине ----
# ДЕНЬГИ: сумма успешных платежей прямо из pay, без единого соединения (как в задаче 1).
money_control = float(pay.loc[pay["status"] == "success", "amount"].sum())
# ЛЮДИ: отдельный запрос к базе — владелец или участник хотя бы одной подписки.
people_control = int(q(f"""
    select count(*) as people
    from prime.clients c
    where {CLIENT_FILTER}
      and (exists (select 1 from prime.subscriptions s where s.owner_client_id = c.client_id)
        or exists (select 1 from prime.subscription_members m where m.client_id = c.client_id))
""").iloc[0, 0])

answer_13 = print_protocol(protocol, mart, N_DECLARED,
                           money_mart=float(mart["paid"].sum()), money_control=money_control,
                           people_mart=int(mart["uses_prime"].sum()), people_control=people_control)
answer_13

**Вопрос 4.** Где ваш протокол заранее ждал потерь ключей и почему это не ошибка — назовите числа. Что стало бы со строками ЗЕРНО и ЛЮДИ, если бы участия не свернули до клиента? Посчитайте, а не угадывайте.

In [ ]:
# Числа для ответа на вопрос 4 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.

# Где протокол заранее ждал потерь ключей — те же числа, что объявлены в задаче 13.
print("ожидали потерь: клиент->подписка", EXPECT_NO_SUB,
      "| платежи->клиент", EXPECT_NO_PAY, "| участия->клиент", EXPECT_NO_MEMBER)

# Ломаем намеренно: присоединяем участия БЕЗ свёртки до клиента.
broken = (mart.drop(columns="n_memberships")
          .merge(members.rename(columns={"subscription_id": "member_of"}), on="client_id", how="left"))
broken_uses = broken["subscription_id"].notna() | broken["member_of"].notna()

print()
print(f"ЗЕРНО   строк {len(broken)}, клиентов {broken['client_id'].nunique()}, объявляли {N_DECLARED}"
      f"   -> {'ok' if len(broken) == N_DECLARED else 'ВНИМАНИЕ'}")
print(f"ЛЮДИ    по сломанной витрине {int(broken_uses.sum())}, независимо {people_control}"
      f"   -> {'ok' if int(broken_uses.sum()) == people_control else 'ВНИМАНИЕ'}")
print(f"ДЕНЬГИ  по сломанной витрине {broken['paid'].sum():,.2f} руб. против {money_control:,.2f}".replace(",", " "))
print("мест участия (строк members):", len(members), " людей-участников:", len(memberships))

#### ✍️ Ответ на вопрос 4

Протокол ожидал потерь при джоинах, и это не ошибка, а логика фильтрации:

- 0 на шаге «клиент - подписка», потому что по условию сегмента у всех есть подписка.
- 4 007 на шаге «платежи - клиент» — это те самые люди из задачи 2, у которых есть подписка, но нет успешных платежей за окно.
- 16 257 на шаге «участия - клиент» — это пользователи, которые не сидят в чужих подписках.

Если не свернуть участия до клиента перед джоином, у нас задублируются строки. Вместо 49 541 уникального клиента мы получим 71 356 строк. Зерно витрины сломается, метрика ЛЮДИ покажет 71 356 человек, а деньги раздуются до 104,5 млн руб. вместо 72,5 млн. Это происходит потому, что мест участия (55 099) больше, чем самих людей-участников (33 284). Поэтому необходимо сворачивать, чтобы считать реальных людей.

## Часть 6. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 14)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» после каждого уровня.")
print("\n--- служебная строка, не удаляйте ---")
print("HW2_ANSWERS=" + json.dumps(
    {"client_filter": CLIENT_FILTER, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 7. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–6 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: в нём две части, каждая до 1 балла сверху. Они засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется у преподавателя целиком — в том числе на контрольном сегменте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Загруженных таблиц для исследования хватает: `pay`, `subs` и `tariffs`. Считайте от них, а не от чисел, вписанных руками, — на контрольном сегменте код должен отработать без правок.

### Исследование А. Что подорожание дало выручке

С 1 марта 2026 года «Базовый» подорожал с 199 до 249 ₽, «Расширенный» — с 399 до 449 ₽. Финансы сравнили выручку сегмента за март–июнь с январём–февралём и записали весь рост в заслугу подорожания. Руководитель просит проверить, сколько рублей в месяц на самом деле дала цена.

**Что сделать**

1. По месяцам окна посчитайте выручку и число успешных платежей — по каждому тарифу.
2. Разделите изменение выручки после 1 марта на две части. **Цена** — разница между тем, сколько стоили платежи марта–июня, и тем, сколько те же платежи стоили бы по старой цене. **Объём** — всё остальное.
3. Учтите две вещи, которые меняют число платежей в месяц без всякого подорожания. Подписка списывается раз в 30 дней, а месяцы разной длины. И в данных апреля есть дыра — вы видели её в вопросе 2.
4. Ответьте: сколько рублей в месяц приносит подорожание вашему сегменту — и насколько ошибается наивное сравнение «март–июнь против января–февраля».

In [ ]:
# Исследование А: ваш код. Всё — от загруженных таблиц.
# Вопрос: сколько рублей в месяц сегменту приносит подорожание 1 марта 2026,
# и насколько ошибается наивное сравнение «март–июнь против января–февраля»?

def rub(x: float) -> str:
    """Рубли с пробелом между разрядами: 1234567 -> «1 234 567»."""
    return f"{x:,.0f}".replace(",", " ")


PRICE_DAY = pd.Timestamp("2026-03-01", tz="UTC")

# Старая и новая цена каждого тарифа — из починенного справочника задачи 6, не руками.
def price_on(moment: pd.Timestamp) -> pd.Series:
    valid = ref_fixed[(ref_fixed["valid_from"] <= moment) & (moment < ref_fixed["valid_to"] + pd.Timedelta(days=1))]
    return valid.set_index("tariff_id")["monthly_fee"]

old_price = price_on(PRICE_DAY - pd.Timedelta(days=1))
new_price = price_on(PRICE_DAY)
print("цены до и после 1 марта:", pd.DataFrame({"old": old_price, "new": new_price}).to_dict("index"))

# 1. По месяцам и тарифам: выручка и число успешных платежей.
ra = (pay[pay["status"] == "success"]
      .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
ra["month"] = ra["paid_at"].dt.strftime("%Y-%m")
monthly = ra.groupby(["month", "tariff_id"]).agg(revenue=("amount", "sum"), n=("payment_id", "size")).reset_index()

# Сколько дней каждого месяца реально есть в данных (дыра в апреле!) — по всем попыткам списания.
days_in_data = pay.groupby(pay["paid_at"].dt.strftime("%Y-%m"))["paid_at"].agg(lambda s: s.dt.date.nunique())
monthly["days"] = monthly["month"].map(days_in_data)

# 2. Цена — сколько платежи стоили сверх старой цены. Объём — всё остальное.
monthly["old"] = monthly["tariff_id"].map(old_price)
monthly["price_part"] = monthly["revenue"] - monthly["n"] * monthly["old"]   # до марта это 0
# 3. Нормировка: платёж раз в 30 дней, поэтому меряем всё «на 30 дней данных».
monthly["revenue_30d"] = monthly["revenue"] / monthly["days"] * 30
monthly["price_30d"] = monthly["price_part"] / monthly["days"] * 30
print()
print(monthly.round(0).to_string(index=False))

total = monthly.groupby("month")[["revenue", "price_part", "revenue_30d", "price_30d"]].sum()
before_m = total.index < "2026-03"
after_m = ~before_m

naive = total.loc[after_m, "revenue"].mean() - total.loc[before_m, "revenue"].mean()
naive_30d = total.loc[after_m, "revenue_30d"].mean() - total.loc[before_m, "revenue_30d"].mean()
price_effect = total.loc[after_m, "price_30d"].mean()
volume_effect = naive_30d - price_effect

print()
print(f"наивно (средний месяц мар–июн минус янв–фев): {rub(naive)} руб.")
print(f"то же на 30 дней данных:                       {rub(naive_30d)} руб.")
print(f"  из них цена (подорожание), на 30 дней:       {rub(price_effect)} руб.")
print(f"  из них объём (меньше платежей), на 30 дней:  {rub(volume_effect)} руб.")
print(f"наивное сравнение приписывает цене {rub(naive)} руб. вместо {rub(price_effect)} руб.")

#### ✍️ Исследование А

Вопрос: сколько рублей в месяц дало подорожание 1 марта 2026 года.

Ход проверки такой: выручку с марта по июнь разделили на «эффект цены» (сколько платежи стоили поверх старой цены) и «эффект объёма» (всё остальное). Чтобы избежать искажений из-за разной длины месяцев и дыры в данных 14-15 апреля, всё пересчитали на 30 дней данных.

В результате наивное сравнение выручки (март–июнь против января–февраля) показывает рост на 861 508 руб., а при нормировке на 30 дней — +678 668 руб. Но если заглянуть внутрь, то за счёт самого подорожания (цена) мы заработали сверху 1 904 559 руб. А вот за счёт снижения объёма платежей потеряли 1 225 892 руб. (наша база тает из-за оттока).

Наивный метод сильно занижает реальный вклад подорожания (более чем в два раза), так как падение объёма базы маскирует профит от повышения цен.

Чего расчёт не доказывает: нельзя утверждать, что падение объёма никак не связано с подорожанием (возможно, кто-то ушёл именно из-за цены), и нет гарантий, что по старым прайсам мы провели бы ровно столько же платежей.


### Исследование Б. Стоило ли подорожание подписчиков

Маркетинг опасается, что подорожание вызвало волну отмен, и предлагает вернуть старые цены. Проверьте на своём сегменте, есть ли для этого основания.

**Что сделать**

1. По месяцам с января по июнь посчитайте отток: сколько подписок закончилось в месяце из тех, что были активны на его начало. Данные — `subs`, колонки `started_at` и `ended_at`.
2. Сравните отток до и после 1 марта — отдельно для «Базового» (+25 %) и «Расширенного» (+12,5 %). Если бы цена отпугивала, какой тариф должен был отреагировать сильнее?
3. Посмотрите на причины отмен: как менялась доля `too_expensive` среди отмен по месяцам.
4. Вывод для маркетинга: есть ли в данных след подорожания в оттоке — и чего ваш расчёт не доказывает. Сколько у вас отмен в месяц и хватит ли этого, чтобы заметить эффект?

In [ ]:
# Исследование Б: ваш код. Всё — от загруженных таблиц.
# Вопрос: вызвало ли подорожание 1 марта волну отмен подписок сегмента?

rows = []
for month_start in pd.date_range("2026-01-01", "2026-06-01", freq="MS", tz="UTC"):
    month_end = month_start + pd.offsets.MonthBegin(1)
    # Активна на начало месяца: началась раньше и не закончилась до начала месяца.
    active = subs[(subs["started_at"] < month_start)
                  & (subs["ended_at"].isna() | (subs["ended_at"] >= month_start))]
    ended = active[(active["ended_at"] >= month_start) & (active["ended_at"] < month_end)]
    for tariff_id, group in active.groupby("tariff_id"):
        gone = ended[ended["tariff_id"] == tariff_id]
        rows.append({"month": month_start.strftime("%Y-%m"), "tariff_id": tariff_id,
                     "active": len(group), "ended": len(gone),
                     "too_expensive": int((gone["cancel_reason"] == "too_expensive").sum())})

churn = pd.DataFrame(rows)
churn["churn_pct"] = churn["ended"] / churn["active"] * 100
churn["too_expensive_share"] = churn["too_expensive"] / churn["ended"] * 100
print(churn.round(2).to_string(index=False))

# До и после 1 марта — по каждому тарифу. «Базовый» подорожал на 25 %, «Расширенный» — на 12,5 %.
churn["period"] = churn["month"].map(lambda m: "до 1 марта" if m < "2026-03" else "после")
summary = churn.groupby(["tariff_id", "period"]).agg(active=("active", "sum"), ended=("ended", "sum"),
                                                     too_expensive=("too_expensive", "sum"))
summary["churn_pct_month"] = summary["ended"] / summary["active"] * 100
summary["too_expensive_share"] = summary["too_expensive"] / summary["ended"] * 100
# Насколько точно мы меряем месячный отток: стандартная ошибка доли, в п.п.
p = summary["ended"] / summary["active"]
summary["se_pp"] = (p * (1 - p) / summary["active"]) ** 0.5 * 100
print()
print(summary.round(2).to_string())
print()
print("отмен в месяц по сегменту:", churn.groupby("month")["ended"].sum().to_dict())

#### ✍️ Исследование Б

Вопрос: вызвало ли подорожание 1 марта 2026 года волну отмен.

Ход проверки такой: посчитали месячный отток (закончились в месяце / активны на начало месяца) в разрезе тарифов до и после 1 марта. Также посмотрели динамику доли причины too_expensive. По итогу на «Базовом» тарифе (который подорожал сильнее всего, на 25%) отток практически не изменился, 3,88% - 3,87%. На «Расширенном» (подорожал на 12,5%) отток немного подрос с 3,64% до 3,83%, но это на грани статистической погрешности. Если бы цена пугала людей, Базовый тариф отреагировал бы гораздо сильнее, но этого не произошло. Доля отмен по причине too_expensive тоже болтается без явного тренда, на Базовом выросла с 22,1% до 23,3%, а на Расширенном даже упала с 25,5% до 24,6%.
Вывод для маркетинга можно сделать следующий: в данных нет абсолютно никаких следов того, что подорожание вызвало волну оттока. Оснований откатывать цены назад сейчас нет.

Чего расчёт не доказывает: эффект отмены может быть отложенным (люди выключат подписку только перед следующим списанием). Также мы исследуем только плательщиков с кошелька, а не всех пользователей, и у нас нет контрольной группы, которой цены не поднимали, чтобы сравнить чистый эффект.